# 🧩 DU Config Timing Block Embedding Tool
This notebook extracts timing-related config blocks from the DU `.conf` and enables semantic search via embedding.

In [ ]:
!pip install sentence-transformers chromadb

In [ ]:
import re

with open("sample_en.conf", "r") as f:
    config_text = f.read()

# Find blocks that include timing keywords
timing_keys = ["Tadv", "T2a", "T1a", "Ta3", "Ta4"]
blocks = []
current_block = []
in_block = False

for line in config_text.splitlines():
    if "{" in line:
        current_block = [line]
        in_block = True
    elif "}" in line and in_block:
        current_block.append(line)
        block_text = "\n".join(current_block)
        if any(k in block_text for k in timing_keys):
            blocks.append(block_text)
        in_block = False
    elif in_block:
        current_block.append(line)

print(f"🧩 Found {len(blocks)} timing-related blocks. Example:")
print(blocks[0][:500])


In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

model = SentenceTransformer("all-MiniLM-L6-v2")

# Init Chroma in-memory DB
chroma_client = chromadb.Client(Settings(chroma_db_impl="duckdb+parquet", persist_directory=None))
collection = chroma_client.create_collection(name="timing_blocks")

embeddings = model.encode(blocks).tolist()
for i, block in enumerate(blocks):
    collection.add(documents=[block], ids=[f"block_{i}"], embeddings=[embeddings[i]])

print("✅ Embedded all blocks.")


In [ ]:
# Run semantic search
query = "uplink delay T2a_up too long"
query_vec = model.encode([query]).tolist()[0]

results = collection.query(query_embeddings=[query_vec], n_results=2)

print("🔍 Top matching config blocks:")
for i, res in enumerate(results["documents"][0]):
    print(f"\n--- Match #{i+1} ---\n{res}")
